# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a line chart because the question focuses on growth over time relative to a starting point. A line chart makes it easy to compare growth trends across continents and identify which continent is growing fastest.

In [10]:
# Q1
# YOUR CODE HERE
continent_pop = (
    df_long.groupby(["Continent", "Year"], as_index=False)["Population"]
    .sum()
)

base = continent_pop[continent_pop["Year"] == 1970][["Continent", "Population"]]
base = base.rename(columns={"Population": "BasePop"})

continent_pop = continent_pop.merge(base, on="Continent")
continent_pop["Growth_Index"] = (
    continent_pop["Population"] / continent_pop["BasePop"] * 100
)

fig = px.line(
    continent_pop,
    x="Year",
    y="Growth_Index",
    color="Continent",
    title="Africa has grown fastest relative to its 1970 population"
)

fig.update_layout(template="plotly_white")

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a sorted horizontal bar chart because the goal is to compare countries by the number of people added between 1970 and 2022. Bar charts make ranking and comparison straightforward.

In [11]:
# Q2
# YOUR CODE HERE
growth = df[["Country", "Continent", "1970 Population", "2022 Population"]].copy()

growth["Added"] = growth["2022 Population"] - growth["1970 Population"]

top15 = growth.sort_values("Added", ascending=False).head(15)

fig = px.bar(
    top15.sort_values("Added"),
    x="Added",
    y="Country",
    orientation="h",
    color="Continent",
    title=f"{(top15['Continent']=='Africa').sum()} of the 15 biggest population gains are in Africa"
)

fig.update_layout(template="plotly_white")

fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a scatter plot because the question asks about the relationship between land area and population. Scatter plots are the best way to examine correlations and identify outliers.

In [12]:
# Q3
# YOUR CODE HERE
fig = px.scatter(
    df,
    x="Area",
    y="2022 Population",
    color="Continent",
    hover_name="Country",
    log_x=True,
    log_y=True,
    title="Land area and population are related, but countries like India and China stand out"
)

fig.update_layout(template="plotly_white")

fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a box plot because the question asks for the spread of population density within continents, not just averages. Box plots clearly show median values, variation and outliers.

In [13]:
# Q4
# YOUR CODE HERE
fig = px.box(
    df,
    x="Continent",
    y="Density",
    color="Continent",
    title="Asia has the highest population densities and the widest spread"
)

fig.update_layout(template="plotly_white")

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a choropleth map because the question asks where countries grew or shrank and where they are located in the world. A map makes geographic patterns easy to see.

In [14]:
# Q5
# YOUR CODE HERE
df["PctChange"] = (
    (df["2022 Population"] - df["2000 Population"])
    / df["2000 Population"] * 100
)

fig = px.choropleth(
    df,
    locations="CCA3",
    color="PctChange",
    hover_name="Country",
    color_continuous_scale="RdBu",
    title="Population growth since 2000 is strongest across much of Africa and Asia"
)

fig.update_layout(template="plotly_white")

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a treemap because it shows both levels of the hierarchy at the same time: continents and the countries within them. It is ideal for part-to-whole comparisons.

In [15]:
# Q6
# YOUR CODE HERE
fig = px.treemap(
    df,
    path=["Continent", "Country"],
    values="2022 Population",
    title="Most of the world's population lives in Asia"
)

fig.update_layout(template="plotly_white")

fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a 100% stacked area chart because the question is about changing shares of the world population over time. The chart clearly shows gains and losses in share.


In [16]:
# Q7
# YOUR CODE HERE
share_df = df_long.copy()

share_df["Group"] = share_df["Continent"]

share_df.loc[
    ~share_df["Continent"].isin(["Africa", "Asia", "Europe"]),
    "Group"
] = "Rest of the world"

grp = (
    share_df.groupby(["Group", "Year"], as_index=False)["Population"]
    .sum()
)

world = (
    grp.groupby("Year", as_index=False)["Population"]
    .sum()
    .rename(columns={"Population": "WorldPop"})
)

grp = grp.merge(world, on="Year")
grp["Share"] = grp["Population"] / grp["WorldPop"] * 100

fig = px.area(
    grp,
    x="Year",
    y="Share",
    color="Group",
    title="Africa's share of the world's population is steadily increasing"
)

fig.update_layout(template="plotly_white")

fig.show()